In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
pip install tensorflow

In [ ]:
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import categorical_crossentropy #should research
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint #should research

from keras.layers import Convolution2D, MaxPooling2D
from keras.layers import Dense, Flatten
from keras.models import Sequential


import os

from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
import itertools
import shutil
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
input_path = '/content/drive/MyDrive/LENET'
os.listdir(input_path)

['main_dir',
 'Copy of resnet_model.pth',
 '.ipynb_checkpoints',
 'HAM10000_images_part_1',
 'HAM10000_images_part_2',
 'resnet_model.pth']

In [ ]:
import os

main_dir = '/content/drive/MyDrive/LENET/main_dir'

# Create main directories safely
os.makedirs(main_dir, exist_ok=True)

train_dir = os.path.join(main_dir, 'train_dir')
val_dir = os.path.join(main_dir, 'val_dir')
test_dir = os.path.join(main_dir, 'test_dir')

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Class names
classes = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']

# Create class folders for train, val, test
for cls in classes:
    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(test_dir, cls), exist_ok=True)

In [14]:
import os
df_data = pd.read_csv('/content/drive/MyDrive/LENET/HAM10000_metadata.csv')

df_data.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [15]:
df = df_data.groupby('lesion_id').count()

df = df[df['image_id'] == 1]

df.reset_index(inplace=True) #should research

In [16]:
def identify_duplicates(x):

    unique_list = list(df['lesion_id'])

    if x in unique_list:
        return 'no_duplicates'
    else:
        return 'has_duplicates'

df_data['duplicates'] = df_data['lesion_id']

df_data['duplicates'] = df_data['duplicates'].apply(identify_duplicates)

In [17]:
df.shape

(5514, 7)

In [18]:
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

In [19]:
from sklearn.model_selection import KFold, StratifiedKFold

skf = StratifiedKFold(n_splits = 10,random_state = 7, shuffle = True)

In [20]:
df = df_data[df_data['duplicates'] == 'no_duplicates']

y = df['dx']
print(df.shape)

#_, df_val = train_test_split(df, test_size = 0.17, random_state=101, stratify=y)
#should research
df_train, df_test, y_train, y_test = train_test_split(df, y, test_size=0.3627, random_state=1)

df_test, df_val, y_test, y_val = train_test_split(df_test, y_test, test_size=0.5, random_state=1)
print(df_train.shape)
print(df_test.shape)
print(df_val.shape)

(5514, 8)
(3514, 8)
(1000, 8)
(1000, 8)


In [21]:
def identify_val_rows(x):

    val_list = list(df_val['image_id'])
    test_list = list(df_test['image_id'])

    if str(x) in val_list or str(x) in test_list:
        return 'val'
    else:
        return 'train'


df_data['train_or_val'] = df_data['image_id']

df_data['train_or_val'] = df_data['train_or_val'].apply(identify_val_rows)

df_train = df_data[df_data['train_or_val'] == 'train']

In [22]:
df_train

,lesion_id,image_id,dx,dx_type,age,sex,localization,duplicates,train_or_val
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,has_duplicates,train
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,has_duplicates,train
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,has_duplicates,train
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,has_duplicates,train
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,has_duplicates,train
...,...,...,...,...,...,...,...,...,...
10010,HAM_0002867,ISIC_0033084,akiec,histo,40.0,male,abdomen,has_duplicates,train
10011,HAM_0002867,ISIC_0033550,akiec,histo,40.0,male,abdomen,has_duplicates,train
10012,HAM_0002867,ISIC_0033536,akiec,histo,40.0,male,abdomen,has_duplicates,train
10013,HAM_0000239,ISIC_0032854,akiec,histo,80.0,male,face,has_duplicates,train


In [23]:
df_val

,lesion_id,image_id,dx,dx_type,age,sex,localization,duplicates
905,HAM_0005845,ISIC_0025851,bkl,consensus,80.0,female,face,no_duplicates
3857,HAM_0005397,ISIC_0028105,nv,follow_up,35.0,male,lower extremity,no_duplicates
2213,HAM_0006744,ISIC_0031561,mel,histo,55.0,male,back,no_duplicates
3989,HAM_0001909,ISIC_0025481,nv,follow_up,55.0,male,upper extremity,no_duplicates
4028,HAM_0000010,ISIC_0027730,nv,follow_up,70.0,female,back,no_duplicates
...,...,...,...,...,...,...,...,...
6034,HAM_0006589,ISIC_0031187,nv,follow_up,40.0,male,trunk,no_duplicates
3861,HAM_0007285,ISIC_0026718,nv,follow_up,55.0,female,genital,no_duplicates
8105,HAM_0004014,ISIC_0029809,nv,histo,45.0,male,chest,no_duplicates
6426,HAM_0005716,ISIC_0031841,nv,follow_up,50.0,male,back,no_duplicates


In [24]:
df_test

,lesion_id,image_id,dx,dx_type,age,sex,localization,duplicates
5637,HAM_0002248,ISIC_0029011,nv,follow_up,40.0,female,trunk,no_duplicates
8226,HAM_0001212,ISIC_0024846,nv,histo,40.0,female,back,no_duplicates
9403,HAM_0003362,ISIC_0033313,nv,consensus,45.0,female,unknown,no_duplicates
5619,HAM_0001395,ISIC_0031818,nv,follow_up,50.0,female,foot,no_duplicates
1947,HAM_0005600,ISIC_0025520,mel,histo,65.0,male,back,no_duplicates
...,...,...,...,...,...,...,...,...
999,HAM_0004078,ISIC_0026566,bkl,consensus,70.0,female,trunk,no_duplicates
635,HAM_0001486,ISIC_0029384,bkl,histo,40.0,male,face,no_duplicates
2957,HAM_0000807,ISIC_0032164,bcc,histo,65.0,female,chest,no_duplicates
4710,HAM_0000334,ISIC_0026860,nv,follow_up,35.0,female,abdomen,no_duplicates


In [ ]:
import os
import shutil
from tqdm import tqdm

#Ensure image_id is index (safe check)
if 'image_id' in df_data.columns:
    df_data.set_index('image_id', inplace=True)

#Load image folders (FAST using set)
folder_1 = set(os.listdir(os.path.join(input_path, 'HAM10000_images_part_1')))
folder_2 = set(os.listdir(os.path.join(input_path, 'HAM10000_images_part_2')))

#Create lists
train_list = list(df_train['image_id'])
val_list = list(df_val['image_id'])
test_list = list(df_test['image_id'])


#FUNCTION to copy images (avoids repeating code)
def copy_images(image_list, target_dir):

    for image in tqdm(image_list):

        file_name = image + '.jpg'

        #Get label safely
        try:
            label = df_data.loc[image, 'dx']
        except KeyError:
            continue  # skip if not found

        #Check in folder 1
        if file_name in folder_1:
            src = os.path.join(input_path, 'HAM10000_images_part_1', file_name)
            dst = os.path.join(target_dir, label, file_name)
            shutil.copy(src, dst)

        #Check in folder 2
        elif file_name in folder_2:
            src = os.path.join(input_path, 'HAM10000_images_part_2', file_name)
            dst = os.path.join(target_dir, label, file_name)
            shutil.copy(src, dst)


#Run copying
print("Copying TRAIN images...")
copy_images(train_list, train_dir)

print("Copying VALIDATION images...")
copy_images(val_list, val_dir)

print("Copying TEST images...")
copy_images(test_list, test_dir)

Copying TRAIN images...


  3%|▎         | 201/8015 [05:20<3:19:42,  1.53s/it]

**Data Augumentataion**

In [ ]:
class_list = ['mel', 'bkl','bcc','akiec','vasc','df']

for clas in class_list:
    aug_dir = 'aug_dir'
    os.mkdir(aug_dir)

    img_dir = os.path.join(aug_dir, 'img_dir')
    os.mkdir(img_dir)

    img_class = clas

    img_list = os.listdir(train_dir +"/"+ img_class)

    for fname in img_list:

        src = os.path.join(train_dir +"/" + img_class,fname)
#         dst = os.path.join(img_dir,fname)
        dst = img_dir

        shutil.copy(src, dst)

    path = aug_dir
    save_path = train_dir +"/" + img_class

    datagen = ImageDataGenerator(rotation_range=180,width_shift_range=0.1,height_shift_range=0.1,zoom_range=0.1,horizontal_flip=True,vertical_flip=True,fill_mode='nearest')
    batch_size = 50

    aug_datagen = datagen.flow_from_directory(path, save_to_dir = save_path, save_format='jpg',target_size=(224,224),batch_size=batch_size)

    num_aug_imgs_wanted = 6000
    num_files = len(os.listdir(img_dir))
    num_batches = int(np.ceil((num_aug_imgs_wanted - num_files)/batch_size))

    for i in range(0, num_batches):
        imgs, labels = next(aug_datagen) #should research

    shutil.rmtree('aug_dir')

In [ ]:
num_train_samples = 8000
num_val_samples = 1000
num_test_samples = 1000
train_batch_size = 10
val_batch_size = 10
test_batch_size = 10
image_size = 28

train_steps = np.ceil(num_train_samples / train_batch_size)
val_steps = np.ceil(num_val_samples / val_batch_size)
test_steps = np.ceil(num_test_samples / test_batch_size)

In [ ]:
train_dir

In [ ]:
datagen = ImageDataGenerator(preprocessing_function = tf.keras.applications.mobilenet.preprocess_input)

train_batches = datagen.flow_from_directory(train_dir,target_size = (image_size, image_size), batch_size=train_batch_size)

val_batches = datagen.flow_from_directory(val_dir,target_size=(image_size, image_size), batch_size=val_batch_size)

test_batches = datagen.flow_from_directory(test_dir, target_size=(image_size, image_size), batch_size=1,shuffle=False)


In [ ]:
model = Sequential()
model.add(Convolution2D(6, kernel_size=(5, 5), activation='relu', input_shape=(28, 28, 3)))
# model.summary()
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Convolution2D(16, kernel_size=(5, 5), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(120, activation='relu'))
model.add(Dense(84, activation='relu'))
model.add(Dense(7, activation='softmax'))

In [ ]:
model.summary()

In [ ]:
def get_model_name(k):
    return 'model_'+str(k)+'.h5'

In [ ]:
idg = ImageDataGenerator(width_shift_range=0.1,
                         height_shift_range=0.1,
                         zoom_range=0.3,
                         fill_mode='nearest',
                         horizontal_flip = True,
                         rescale=1./255)

In [ ]:
df.columns

In [ ]:
from matplotlib import pyplot
from matplotlib import pyplot as plt
filters, biases = model.layers[0].get_weights()
# normalize filter values to 0-1 so we can visualize them
f_min, f_max = filters.min(), filters.max()
filters = (filters - f_min) / (f_max - f_min)
# plot first few filters
n_filters, ix = 5, 1
for i in range(n_filters):
	# get the filter
	f = filters[:, :, :, i]
	# plot each channel separately
	for j in range(3):
		# specify subplot and turn of axis
		ax = pyplot.subplot(n_filters, 3, ix)
		ax.set_xticks([])
		ax.set_yticks([])
		# plot filter channel in grayscale
		pyplot.imshow(f[:, :, j], cmap='gray')
		ix += 1
# show the figure
pyplot.show()


In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='rmsprop',metrics=["accuracy"])

hist = model.fit(train_batches, steps_per_epoch=int(train_steps), epochs=20, validation_data= val_batches, validation_steps=int(val_steps))

In [ ]:
model.save("model.h5")

In [ ]:
filters, biases = model.layers[0].get_weights()
# normalize filter values to 0-1 so we can visualize them
f_min, f_max = filters.min(), filters.max()
filters = (filters - f_min) / (f_max - f_min)
# plot first few filters
n_filters, ix = 5, 1
for i in range(n_filters):
	# get the filter
	f = filters[:, :, :, i]
	# plot each channel separately
	for j in range(3):
		# specify subplot and turn of axis
		ax = pyplot.subplot(n_filters, 3, ix)
		ax.set_xticks([])
		ax.set_yticks([])
		# plot filter channel in grayscale
		pyplot.imshow(f[:, :, j], cmap='gray')
		ix += 1
# show the figure
pyplot.show()


In [ ]:
from keras.preprocessing.image import load_img
from keras.preprocessing.image import img_to_array
from keras.applications.vgg16 import preprocess_input
from numpy import expand_dims

In [ ]:
predictions = model.predict(test_batches, steps=1000, verbose=1)

In [ ]:
print(predictions)

In [ ]:
test_labels = test_batches.classes

In [ ]:
test_labels

In [ ]:
loss, acc = model.evaluate(test_batches, verbose=1)

In [ ]:
acc

In [ ]:
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig('conf.png')

In [ ]:
confusion_mat = confusion_matrix(test_labels, predictions.argmax(axis=1))

In [ ]:
test_batches.class_indices

In [ ]:
cm_plot_labels = ['akiec', 'bcc', 'bkl', 'df', 'mel','nv', 'vasc']

plot_confusion_matrix(confusion_mat, cm_plot_labels, title='Confusion Matrix')

ResNet

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os

In [ ]:
train_dir = "/content/drive/MyDrive/LENET/main_dir/train_dir"
val_dir = "/content/drive/MyDrive/LENET/main_dir/val_dir"
test_dir = "/content/drive/MyDrive/LENET/main_dir/test_dir"

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet needs 224x224
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [ ]:
train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
model = models.resnet18(pretrained=True)

# Modify final layer
num_classes = len(train_dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
from tqdm import tqdm

epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0
    batch_size = 64
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}]", leave=True)

    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        # Forward
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward
        loss.backward()
        optimizer.step()

        # Loss
        running_loss += loss.item()

        # Accuracy
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # Update progress bar
        loop.set_postfix(
            loss=running_loss / (loop.n + 1),
            acc=100 * correct / total
        )

    # Validation step
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    print(f"\nEpoch {epoch+1} Summary:")
    print(f"Train Loss: {running_loss:.4f}")
    print(f"Train Accuracy: {100 * correct / total:.2f}%")
    print(f"Validation Accuracy: {100 * val_correct / val_total:.2f}%\n")

In [ ]:
model.eval()
test_correct = 0
test_total = 0

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(f"Test Accuracy: {100 * test_correct / test_total:.2f}%")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(all_labels, all_preds)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(all_labels, all_preds))

In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/LENET/resnet_model.pth")